# Model initiation

In [ ]:
import os
import csv
import cv2
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm import tqdm
from glob import glob
from concurrent.futures import ThreadPoolExecutor
from transnetv2_pytorch import TransNetV2
from transformers import AutoModel, AutoProcessor
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity

device = "cuda" if torch.cuda.is_available() else "cpu"
model_transnetv2 = TransNetV2().eval().to(device)
ckpt = "google/siglip2-so400m-patch14-384"
model_siglip2 = AutoModel.from_pretrained(
    ckpt,
    device_map="auto",
    dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2" # Install flash-attn before using this
).eval()
processor = AutoProcessor.from_pretrained(ckpt)

c:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\GITHUB_REPOS\Determinative-HCMCAIC26\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100%|██████████| 888/888 [00:00<00:00, 918.15it/s] 


# Define utils

In [ ]:
class KeyframeAndFeatureExtractor:
    def __init__(self, shot_extraction_model, feature_extraction_model, feature_extraction_processor, save_dir="."):
        self.shot_extraction_model = shot_extraction_model
        self.feature_extraction_model = feature_extraction_model
        self.feature_extraction_processor = feature_extraction_processor
        self.img_dir = os.path.join(save_dir, "imgs")
        self.map_dir = os.path.join(save_dir, "maps")
        self.feat_dir = os.path.join(save_dir, "feats", "siglip2_dense")


    def encode_images(self, imgs: list, batch_size: int = 32):
        feats = []
        for i in range(0, len(imgs), batch_size):
            j = len(imgs) if i + batch_size > len(imgs) else i + batch_size
            batch = imgs[i:j]
            preprocs = self.feature_extraction_processor(images=batch, return_tensors='pt')
            
            with torch.no_grad():
                feat = self.feature_extraction_model.get_image_features(**preprocs.to(device)).pooler_output
                feat = F.normalize(feat)

            feats.append(feat.cpu())
        return torch.vstack(feats).float().numpy()


    def extract_keyframes_from_shots(
        self,
        vid_path,
        confidence_threshold: float = 0.5,
        feature_extraction_batch_size: int = 32
    ):
        imgs = []
        maps = []
        with torch.no_grad():
            scenes = self.shot_extraction_model.detect_scenes(vid_path, threshold=confidence_threshold)

        cap = cv2.VideoCapture(vid_path)
        if not cap: return []
        for scene in scenes:
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_order = scene["start_frame"] + fps // 2
            time_order = frame_order / fps
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_order)
            ret, frame = cap.read()
            if not ret: continue

            imgs.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
            maps.append({
                "frame_order": frame_order,
                "time_order": time_order,
                "fps": fps,
                "vid_path": vid_path
            })
        
        feats = self.encode_images(imgs, batch_size=feature_extraction_batch_size)
        return imgs, maps, feats


    def filter_with_clustering(self, imgs: list, maps: list[dict], feats: np.ndarray, clustering_similarity_threshold: float = 0.85):
        similarity_matrix = cosine_similarity(feats)
        distance_matrix = np.clip(1 - similarity_matrix, 0, 1)

        distance_threshold = 1 - clustering_similarity_threshold
        clustering = AgglomerativeClustering(
            metric="precomputed",
            linkage="average",
            distance_threshold=distance_threshold,
            n_clusters=None
        )
        cluster_labels = clustering.fit_predict(distance_matrix)

        unique_imgs, unique_maps, unique_feats = [], [], []
        n_clusters = len(np.unique(cluster_labels))

        for cluster_id in range(n_clusters):
            idxs = np.where(cluster_labels == cluster_id)[0]
            best_idx = idxs[0]

            if len(idxs) > 1:
                cluster_feats = feats[idxs]
                cluster_centroid = np.mean(cluster_feats, axis=0)
                centroid_similarities = cosine_similarity(cluster_feats, cluster_centroid.reshape(1, -1))
                best_idx = idxs[np.argmax(centroid_similarities)]

            unique_imgs.append(imgs[best_idx])
            unique_maps.append(maps[best_idx])
            unique_feats.append(feats[best_idx])
        
        return unique_imgs, unique_maps, unique_feats


    def save_images_and_features(self, imgs: list, maps: list[dict], feats: np.ndarray):
        imgs = [img for img, _ in sorted(list(zip(imgs, maps)), key=lambda x: x[1]["frame_order"])]
        feats = [feat for feat, _ in sorted(list(zip(feats, maps)), key=lambda x: x[1]["frame_order"])]
        maps = sorted(maps, key=lambda x: x["frame_order"])

        csv_list = []
        header = []
        vid_id = ""
        for idx, (img, map, feat) in enumerate(zip(imgs, maps, feats)):
            vid_id = map["vid_path"].split('\\')[-1].split('.')[0]
            frame_id = f"{(idx + 1):03d}"
            os.makedirs(os.path.join(self.img_dir, vid_id), exist_ok=True)
            os.makedirs(os.path.join(self.feat_dir, vid_id), exist_ok=True)

            img.save(os.path.join(self.img_dir, vid_id, f"{frame_id}.jpg"))
            np.save(os.path.join(self.feat_dir, vid_id, f"{frame_id}.npy"), feat)
            map.pop("vid_path", None)
            header = list(map.keys())
            csv_list.append(list(map.values()))

        # csv_list = sorted(csv_list, key=lambda x: x[0])
        with open(os.path.join(self.map_dir, f"{vid_id}.csv"), 'w', newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(header)
            writer.writerows(csv_list)


    def extract_and_save_keyframes(
        self,
        vid_paths: list[str],
        feature_extraction_batch_size: int = 32,
        shot_confidence_threshold: float = 0.5,
        clustering_similarity_threshold: float = 0.85,
        max_workers: int = 1
    ):
        os.makedirs(self.img_dir, exist_ok=True)
        os.makedirs(self.map_dir, exist_ok=True)
        os.makedirs(self.feat_dir, exist_ok=True)

        def process_single_video(path):
            imgs, maps, feats = self.extract_keyframes_from_shots(
                path,
                confidence_threshold=shot_confidence_threshold,
                feature_extraction_batch_size=feature_extraction_batch_size
            )
            unique_imgs, unique_maps, unique_feats = self.filter_with_clustering(
                imgs, maps, feats,
                clustering_similarity_threshold=clustering_similarity_threshold
            )
            self.save_images_and_features(unique_imgs, unique_maps, unique_feats)

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            for _ in tqdm(
                executor.map(process_single_video, vid_paths),
                total=len(vid_paths),
                desc="Extracting keyframes from video"
            ): pass

In [1]:
from extraction_tools import KeyframeAndFeatureExtractor
from models import TransNetV2Model, SigLIP2Model
from glob import glob

save_dir = r'D:\AIC_NEW_EXTRACTION'
shot_extraction_model = TransNetV2Model(device="cuda")
feature_extraction_model = SigLIP2Model(name="google/siglip2-so400m-patch14-384", device="cuda")

extractor = KeyframeAndFeatureExtractor(
    shot_extraction_model,
    feature_extraction_model,
    save_dir
)

c:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\GITHUB_REPOS\Determinative-HCMCAIC26\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Loading quantized Sn4kehead/TransNetV2
[INFO] Sn4kehead/TransNetV2 is on cuda device
[INFO] Loading quantized google/siglip2-so400m-patch14-384
[INFO] google/siglip2-so400m-patch14-384 is on cuda device


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100%|██████████| 888/888 [00:00<00:00, 1204.63it/s]


# Extraction process

In [6]:
vid_paths = []
for i in [21, 22, 23, 24, 25, 26, 27, 28, 29, 30]:
    vid_paths += glob(fr"E:\AIC_VID_ONLY\vids\Videos_L{i}\*.mp4")

extractor.extract_and_save_keyframes(
    vid_paths,
    feature_extraction_batch_size=32,
    shot_confidence_threshold=0.7,
    clustering_similarity_threshold=0.9,
    max_workers=4
)

Extracting keyframes from video: 100%|██████████| 873/873 [2:32:12<00:00, 10.46s/it]  
